<a href="https://colab.research.google.com/github/apexprit/deception-analysis/blob/main/colab_deception_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️‍♂️ Multimodal Deception Analysis - Google Colab

**Run this entire deception detection system in the cloud.**

This notebook provides a complete environment for detecting deception using:
- **Facial micro-expressions** (Mediapipe Face Mesh)
- **Audio speech patterns** (Librosa)
- **SHAP-based explainability**

## 🚀 Quick Start
1. **Runtime → Run all** (Ctrl+F9).
2. If the session restarts, just click **Run all** again.
3. Scroll to the bottom to find the **Gradio link**.

## 1. Environment Setup

In [ ]:
#@title Install & Fix Mediapipe
print("🛠️ Fixing environment and installing dependencies...")

# Uninstall existing versions that might conflict
!pip uninstall -y -q mediapipe protobuf

# Install stable versions
!pip install -q mediapipe==0.10.0
!pip install -q protobuf==3.20.3
!pip install -q opencv-python librosa gradio shap joblib

print("✅ Installation complete. Restarting session to apply changes...")

import os
try:
    # This will restart the Colab runtime automatically
    os.kill(os.getpid(), 9)
except:
    pass

In [ ]:
#@title Clone & Setup Path
import os
import sys
import shutil

if os.path.exists('/content/deception-analysis'):
    shutil.rmtree('/content/deception-analysis')

!git clone -q https://github.com/apexprit/deception-analysis.git /content/deception-analysis
sys.path.insert(0, '/content/deception-analysis')

print("✅ Project ready!")

## 2. Train & Launch App

In [ ]:
#@title Generate Model & Start Web App
import sys
import os
import pandas as pd
import numpy as np
sys.path.insert(0, '/content/deception-analysis')

from src.utils.synthetic_data import SyntheticDataGenerator
from src.utils.config import get_default_config
from src.model.classifier import DeceptionClassifier
import gradio as gr
from src.pipeline import DeceptionPipeline

config = get_default_config()

print("🔧 Generating synthetic dataset for model training...")
generator = SyntheticDataGenerator(config, seed=42)
features_df, labels, _ = generator.generate_dataset(n_truthful=300, n_deceptive=300)

print("📈 Training classifier...")
classifier = DeceptionClassifier(config.model)

# Correctly prepare features and extract names before training
X, feature_names = classifier.prepare_features(features_df)
metrics = classifier.train(X, labels, feature_names)
print(f"✅ Training complete! Accuracy: {metrics.get('train_accuracy', 0):.3f}")

model_dir = '/content/deception-analysis/models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'deception_model.pkl')
classifier.save(model_path)
print(f"✅ Model saved to {model_path}")

print("\n🚀 Launching Deception Analysis UI...")
pipeline = DeceptionPipeline(config)
pipeline.load_model(model_path)

def analyze(video, subject_id):
    if not video: return "Please upload a video.", None, None
    
    output_results = '/content/results'
    os.makedirs(output_results, exist_ok=True)
    
    res = pipeline.analyze_video(
        video, 
        subject_id=subject_id or "S01", 
        generate_visualizations=True, 
        output_dir=output_results
    )
    
    if "error" in res:
        return f"### Error during analysis:\n{res['error']}", None, None
        
    summary = f"### Result: {res['prediction'].upper()}\n- Prob: {res['deception_probability']:.3f}\n- Conf: {res['confidence']:.3f}"
    viz = res.get('visualizations', {})
    
    return summary, viz.get('temporal_trajectory'), viz.get('shap_summary')

with gr.Blocks() as demo:
    gr.Markdown("# 🕵️‍♂️ Multimodal Deception Analysis")
    gr.Markdown("Analyze facial micro-expressions and audio patterns for deception detection.")
    
    with gr.Row():
        with gr.Column():
            v_in = gr.Video(label="Interview Video")
            s_in = gr.Textbox(label="Subject ID", placeholder="e.g. S01")
            btn = gr.Button("Analyze Video", variant="primary")
        with gr.Column():
            sum_out = gr.Markdown(label="Assessment Summary")
            with gr.Tabs():
                with gr.TabItem("Temporal"): 
                    traj_out = gr.Image(label="Probability Trajectory")
                with gr.TabItem("Explainability"): 
                    shap_out = gr.Image(label="SHAP Feature Importance")
    
    btn.click(analyze, [v_in, s_in], [sum_out, traj_out, shap_out])

demo.launch(share=True, debug=True)